# 作业 1：复现 KGW 与 Unbiased 两种 LLM 水印方法

## 研究目标

本实验基于 **Kirchenbauer et al. (2023)** 的 KGW 水印算法和 **Hu et al. (2024)** 的 MarkLLM 评估框架，
在 Qwen2.5-0.5B-Instruct 模型上复现并系统比较两种文本水印方法：KGW 绿名单水印和 Unbiased 分布保持水印。

## 评估维度

参照 MarkLLM 工具包论文 (Pan et al., EMNLP 2024) 和 ACL 2024 LLM Watermarking Tutorial 的建议，本实验从以下维度评估：

1. **检测有效性 (Detection Effectiveness)**：通过 Z-score 量化水印信号强度，Z > 2 视为可检测
2. **文本质量 (Text Quality)**：使用模型自身困惑度 (Perplexity, PPL) 衡量水印对生成质量的影响
3. **计算开销 (Computational Cost)**：记录每种方法的生成延迟

## 参考文献

- Kirchenbauer, J. et al. "A Watermark for Large Language Models." ICML 2023.
- Pan, L. et al. "MarkLLM: An Open-Source Toolkit for LLM Watermarking." EMNLP 2024.
- Thickstun, J. "Robust Distortion-free Watermarks for Language Models." TMLR 2023.
- LLM Watermarking Tutorial, ACL 2024. https://leililab.github.io/llm_watermark_tutorial/


## 1. 环境配置与水印算法实现

### 1.1 算法原理

**KGW 水印** (Kirchenbauer et al., 2023)：在生成每个 token 时，根据前文上下文通过哈希函数将词表划分为
绿名单 (green list, 比例 $\gamma$) 和红名单 (red list)。对绿名单中的 token logit 增加偏置 $\delta$，
使模型倾向于选择绿名单中的 token。检测时统计生成文本中绿名单 token 的比例，通过 Z-test 判断是否嵌入水印。

**Unbiased 水印**：受 distortion-free 水印 (Thickstun, 2023) 启发，不直接改变 logit，而是对概率分布
进行温和重加权：将绿名单 token 的概率乘以因子 $\alpha$ 后重新归一化，以减小对生成分布的扭曲。

### 1.2 实验设置

- 模型：Qwen2.5-0.5B-Instruct (约 5 亿参数)
- 硬件：NVIDIA RTX 4090D (24GB VRAM)
- Prompt 数量：100 条中文 prompt，涵盖科学技术、哲学社会、文学艺术等 12 个领域
- 生成参数：temperature=0.9, top_p=0.95, max_new_tokens=80
- 随机种子：2026（保证可重复性）


In [1]:
import os, json, math, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'
})

# ---- Paths ----
MODEL_DIR = Path('/root/autodl-tmp/Qwen2.5-0.5B-Instruct')
RUN_DIR = Path.cwd()
SUBMISSION_DIR = RUN_DIR.parent if RUN_DIR.name == 'notebooks' else RUN_DIR / 'submission'
DATA_DIR = SUBMISSION_DIR / 'data'
OUT_DIR = SUBMISSION_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Config ----
N_PROMPTS = 100
MAX_NEW_TOKENS = 80
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print(f'Device: {DEVICE}')
print(f'Model path: {MODEL_DIR}')
print(f'Model exists: {MODEL_DIR.exists()}')
print(f'Output dir: {OUT_DIR}')
print(f'Using {N_PROMPTS} prompts')

# ---- Hashing ----
def stable_hash(value):
    return int(hashlib.sha256(str(value).encode('utf-8')).hexdigest(), 16) % (2**32)

# ---- Green list mask ----
def greenlist_mask(prev_tokens, vocab_size, gamma=0.5, hash_key=42, device='cpu'):
    if isinstance(prev_tokens, torch.Tensor):
        prev_tokens = prev_tokens.detach().cpu().tolist()
    context = tuple(prev_tokens[-2:]) if len(prev_tokens) >= 2 else tuple(prev_tokens) or (0,)
    gen = torch.Generator(device=device)
    gen.manual_seed(stable_hash((context, hash_key)))
    return torch.rand(vocab_size, generator=gen, device=device) < gamma

# ---- KGW Watermark Processor ----
class KGWLogitsProcessor(LogitsProcessor):
    """KGW-style green/red list watermark via logit bias (Kirchenbauer et al., ICML 2023)."""
    def __init__(self, gamma=0.5, delta=2.0, hash_key=42, device='cpu'):
        self.gamma, self.delta, self.hash_key, self.device = gamma, delta, hash_key, device
    def __call__(self, input_ids, scores):
        vocab_size = scores.shape[-1]
        for b in range(input_ids.shape[0]):
            mask = greenlist_mask(input_ids[b, -1:], vocab_size, self.gamma, self.hash_key, self.device)
            scores[b][mask] += self.delta
        return scores

# ---- Unbiased Watermark Processor ----
class UnbiasedLogitsProcessor(LogitsProcessor):
    """Distribution-preserving watermark via probability reweighting (inspired by Thickstun, TMLR 2023)."""
    def __init__(self, gamma=0.5, alpha=2.5, hash_key=42, device='cpu'):
        self.gamma, self.alpha, self.hash_key, self.device = gamma, alpha, hash_key, device
    def __call__(self, input_ids, scores):
        vocab_size = scores.shape[-1]
        for b in range(input_ids.shape[0]):
            probs = torch.softmax(scores[b], dim=-1)
            mask = greenlist_mask(input_ids[b], vocab_size, self.gamma, self.hash_key, self.device)
            p_green = probs[mask].sum()
            if p_green < 1e-6 or p_green > 0.999:
                continue
            new_probs = probs.clone()
            new_probs[mask] *= self.alpha
            new_probs /= new_probs.sum()
            scores[b] = torch.log(new_probs.clamp(min=1e-10))
        return scores

# ---- Z-Score Detector ----
class ZDetector:
    """Z-test detector: measures green-token enrichment vs binomial expectation."""
    def __init__(self, tokenizer, gamma=0.5, hash_key=42, device='cpu', full_context=False):
        self.vocab_size = len(tokenizer)
        self.gamma, self.hash_key, self.device, self.full_context = gamma, hash_key, device, full_context
    def detect(self, tokens, prompt_len):
        if isinstance(tokens, list):
            tokens = torch.tensor(tokens)
        tokens = tokens.to(self.device)
        n = len(tokens) - prompt_len
        if n <= 5:
            return 0.0
        hits = 0
        for i in range(prompt_len, len(tokens)):
            ctx = tokens[:i] if self.full_context else tokens[i-1:i]
            mask = greenlist_mask(ctx, self.vocab_size, self.gamma, self.hash_key, self.device)
            tid = int(tokens[i].item())
            if tid < self.vocab_size and bool(mask[tid]):
                hits += 1
        expected = self.gamma * n
        std = math.sqrt(n * self.gamma * (1 - self.gamma))
        return float((hits - expected) / (std + 1e-6))

# ---- Model Loading ----
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), dtype=DTYPE, trust_remote_code=True).to(DEVICE)
    model.eval()
    return model, tokenizer

# ---- Generation ----
@torch.no_grad()
def generate_one(model, tokenizer, prompt, processor=None, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    prompt_len = inputs.input_ids.shape[-1]
    config = dict(
        max_new_tokens=max_new_tokens, do_sample=True,
        temperature=0.9, top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
    )
    if processor is not None:
        config['logits_processor'] = LogitsProcessorList([processor])
    output = model.generate(**inputs, **config)[0]
    text = tokenizer.decode(output[prompt_len:], skip_special_tokens=True)
    return output.detach().cpu().tolist(), prompt_len, text

# ---- Perplexity ----
def calc_ppl(model, tokenizer, text):
    if not text.strip():
        return float('nan')
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=320).to(DEVICE)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs['input_ids']).loss
    return float(torch.exp(loss).detach().cpu())

print("All functions and classes defined successfully.")


Device: cuda
Model path: /root/autodl-tmp/Qwen2.5-0.5B-Instruct
Model exists: True
Output dir: /root/homework/submission/outputs
Using 100 prompts
All functions and classes defined successfully.


## 2. 数据加载与模型初始化

从 `prompts.json` 加载 100 条中文测试 prompt（涵盖 12 个领域），加载本地 Qwen2.5-0.5B-Instruct 模型。

In [2]:
# Load prompts
with open(DATA_DIR / 'prompts.json', 'r', encoding='utf-8') as f:
    prompts_data = json.load(f)
    prompts = prompts_data['test_prompts'][:N_PROMPTS]

print(f'Loaded {len(prompts)} prompts')
print(f'Prompt length range: {min(len(p) for p in prompts)} - {max(len(p) for p in prompts)} chars')
print(f'Average prompt length: {np.mean([len(p) for p in prompts]):.0f} chars')

# Load model
model, tokenizer = load_model()
print(f'Tokenizer vocabulary size: {len(tokenizer):,}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')
print(f'GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')


Loaded 100 prompts
Prompt length range: 27 - 70 chars
Average prompt length: 45 chars


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizer vocabulary size: 151,665
Model parameters: 0.49B
GPU memory allocated: 0.93 GB


## 3. 批量生成实验

对每条 prompt 分别使用三种模式生成：
- **Baseline**：无水印（对照组）
- **KGW**：绿名单偏置水印 ($\gamma=0.5$, $\delta=2.0$)
- **Unbiased**：分布保持水印 ($\gamma=0.5$, $\alpha=2.5$)

对每种生成结果计算 Z-score、PPL 和生成耗时。

In [3]:
# Configure processors and detectors
processors = {
    'baseline': None,
    'kgw': KGWLogitsProcessor(gamma=0.5, delta=2.0, device=DEVICE),
    'unbiased': UnbiasedLogitsProcessor(gamma=0.5, alpha=2.5, device=DEVICE),
}
detectors = {
    'kgw': ZDetector(tokenizer, gamma=0.5, device=DEVICE),
    'unbiased': ZDetector(tokenizer, gamma=0.5, device=DEVICE, full_context=True),
}

rows = []
start_time = time.perf_counter()
for prompt_id, prompt in enumerate(tqdm(prompts, desc='Generating')):
    for method, processor in processors.items():
        t0 = time.perf_counter()
        tokens, prompt_len, text = generate_one(model, tokenizer, prompt, processor=processor)
        detector = detectors['unbiased'] if method == 'unbiased' else detectors['kgw']
        rows.append({
            'prompt_id': prompt_id,
            'method': method,
            'prompt_len': prompt_len,
            'new_tokens': len(tokens) - prompt_len,
            'z_score': detector.detect(tokens, prompt_len),
            'ppl': calc_ppl(model, tokenizer, text),
            'seconds': time.perf_counter() - t0,
            'text_preview': text[:180].replace('\n', ' '),
        })

total_time = time.perf_counter() - start_time
df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / 'assignment1_full_results.csv', index=False, encoding='utf-8-sig')
print(f'\nTotal runtime: {total_time:.1f} seconds ({total_time/60:.1f} minutes)')
print(f'Generated {len(df)} samples ({len(df)//3} prompts x 3 methods)')
print(f'\nFirst 9 rows:')
display(df.head(9))


Generating:   0%|          | 0/100 [00:00<?, ?it/s]


Total runtime: 524.0 seconds (8.7 minutes)
Generated 300 samples (100 prompts x 3 methods)

First 9 rows:


,prompt_id,method,prompt_len,new_tokens,z_score,ppl,seconds,text_preview
0,0,baseline,37,80,0.894427,9.566607,2.942943,《AI驱动的日常生活》 在未来的城市中，人们的生活方式将发生深刻的变化。随着人工智能技术...
1,0,kgw,37,80,5.813775,24.908052,1.680171,【城市中的未来：一家普通家庭的日常生活】 在未来的都市中，AI正逐渐融入到我们的生活之中...
2,0,unbiased,37,80,1.565247,11.011100,1.666346,《科技之光：人工智能在未来的城市中的生活》 在这个快节奏的现代生活中，我们常常被各种琐碎...
3,1,baseline,37,80,1.565247,6.915399,1.668022,黑洞是一种极其密集的天体，它的质量非常大，但体积却非常小，因此其引力极强以至于连光也无法逃...
4,1,kgw,37,80,4.695742,9.590268,1.676998,黑洞是由大质量天体（如中子星、超新星爆炸后的一部分）由于质量巨大而产生的巨大引力，使得任何...
5,1,unbiased,37,80,4.024921,9.922746,1.791784,黑洞是天文学中一种神秘而又令人向往的现象。首先需要指出的是，黑洞并不是由我们日常生活中的物...
6,2,baseline,35,80,1.565247,5.456815,1.733338,最初，量子计算机是通过使用量子比特（qubits）来实现的。一个 qubit 可以同时表示...
7,2,kgw,35,80,5.366562,14.843297,1.751961,《量子计算：理论原理与应用》 随着科技的发展和新材料的涌现，计算机系统已经从二进制位逐步...
8,2,unbiased,35,80,0.223607,5.785895,1.755266,量子计算是一种基于量子力学原理的新型计算机技术，它的基本原理是将信息处理方式从传统的二进制...


## 4. 统计分析

### 4.1 汇总统计

计算三种方法的 Z-score、PPL 和生成时间的均值、标准差、中位数，并进行独立样本 t 检验以评估差异的统计显著性。

In [4]:
# Summary statistics
summary = df.groupby('method').agg(
    samples=('prompt_id', 'count'),
    mean_z=('z_score', 'mean'),
    std_z=('z_score', 'std'),
    median_z=('z_score', 'median'),
    min_z=('z_score', 'min'),
    max_z=('z_score', 'max'),
    mean_ppl=('ppl', 'mean'),
    std_ppl=('ppl', 'std'),
    median_ppl=('ppl', 'median'),
    mean_seconds=('seconds', 'mean'),
    std_seconds=('seconds', 'std'),
    mean_new_tokens=('new_tokens', 'mean'),
).reset_index()

summary.to_csv(OUT_DIR / 'assignment1_full_summary.csv', index=False, encoding='utf-8-sig')
print("Summary Statistics:")
display(summary.round(3))

# Statistical tests
print("\n--- Statistical Significance Tests ---")
methods_list = ['baseline', 'kgw', 'unbiased']
for metric, metric_name in [('z_score', 'Z-score'), ('ppl', 'PPL')]:
    print(f"\n{metric_name} pairwise t-tests (Bonferroni corrected):")
    for i, m1 in enumerate(methods_list):
        for m2 in methods_list[i+1:]:
            v1 = df[df['method']==m1][metric]
            v2 = df[df['method']==m2][metric]
            t_stat, p_val = stats.ttest_ind(v1, v2)
            significant = "***" if p_val < 0.001/3 else ("**" if p_val < 0.01/3 else ("*" if p_val < 0.05/3 else "n.s."))
            print(f"  {m1} vs {m2}: t={t_stat:.3f}, p={p_val:.2e} {significant}")
            if m1 == 'baseline' and m2 == 'kgw':
                cohens_d = (v2.mean() - v1.mean()) / np.sqrt((v1.var() + v2.var())/2)
                print(f"    Cohen's d = {cohens_d:.3f}")


Summary Statistics:


,method,samples,mean_z,std_z,median_z,min_z,max_z,mean_ppl,std_ppl,median_ppl,mean_seconds,std_seconds,mean_new_tokens
0,baseline,100,0.134,1.108,0.224,-2.683,2.236,9.239,3.026,8.908,1.731,0.145,80.0
1,kgw,100,5.217,0.950,5.367,2.460,7.826,12.178,3.940,11.515,1.742,0.060,80.0
2,unbiased,100,2.746,0.947,2.683,0.224,4.919,9.907,3.568,9.209,1.766,0.060,80.0



--- Statistical Significance Tests ---

Z-score pairwise t-tests (Bonferroni corrected):
  baseline vs kgw: t=-34.831, p=2.22e-86 ***
    Cohen's d = 4.926
  baseline vs unbiased: t=-17.923, p=2.53e-43 ***
  kgw vs unbiased: t=18.426, p=8.16e-45 ***

PPL pairwise t-tests (Bonferroni corrected):
  baseline vs kgw: t=-5.914, p=1.44e-08 ***
    Cohen's d = 0.836
  baseline vs unbiased: t=-1.427, p=1.55e-01 n.s.
  kgw vs unbiased: t=4.272, p=3.01e-05 ***


### 4.2 可视化分析

采用多面板图表展示：(a) Z-score 分布对比，(b) PPL 分布对比，(c) Z-score vs PPL 散点图，
(d) 平均生成时间对比，(e) Z-score 累积分布函数 (CDF)，(f) 相关性热力图。

In [5]:
# Enhanced visualization with 6-panel layout
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
methods = ['baseline', 'kgw', 'unbiased']
colors = {'baseline': '#4C78A8', 'kgw': '#F58518', 'unbiased': '#54A24B'}
labels = {'baseline': 'Baseline (No WM)', 'kgw': 'KGW (δ=2.0)', 'unbiased': 'Unbiased (α=2.5)'}

# (a) Z-score distribution
ax = axes[0, 0]
for method in methods:
    sub = df[df['method'] == method]
    ax.hist(sub['z_score'], bins=20, alpha=0.6, label=labels[method], color=colors[method], edgecolor='white')
ax.axvline(2.0, color='black', linestyle='--', linewidth=1.5, label='Detection threshold (Z=2)')
ax.axvline(0.0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_title('(a) Z-score Distribution', fontweight='bold')
ax.set_xlabel('Z-score')
ax.set_ylabel('Frequency')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (b) PPL distribution
ax = axes[0, 1]
for method in methods:
    sub = df[df['method'] == method]
    ax.hist(sub['ppl'], bins=20, alpha=0.6, label=labels[method], color=colors[method], edgecolor='white')
ax.set_title('(b) Perplexity (PPL) Distribution', fontweight='bold')
ax.set_xlabel('PPL')
ax.set_ylabel('Frequency')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (c) PPL vs Z-score scatter with regression
ax = axes[0, 2]
for method in methods:
    sub = df[df['method'] == method]
    ax.scatter(sub['ppl'], sub['z_score'], label=labels[method], color=colors[method], alpha=0.7, edgecolors='white', s=50)
    # Add regression line
    valid = sub['ppl'].notna() & np.isfinite(sub['ppl'])
    if valid.sum() > 5:
        m, b = np.polyfit(sub.loc[valid, 'ppl'], sub.loc[valid, 'z_score'], 1)
        xs = np.linspace(sub.loc[valid, 'ppl'].min(), sub.loc[valid, 'ppl'].max(), 50)
        ax.plot(xs, m*xs + b, color=colors[method], linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(2.0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_title('(c) PPL vs Z-score Trade-off', fontweight='bold')
ax.set_xlabel('Perplexity (PPL)')
ax.set_ylabel('Z-score')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (d) Average generation time
ax = axes[1, 0]
bars = ax.bar([labels[m] for m in methods], 
              [summary[summary['method']==m]['mean_seconds'].values[0] for m in methods],
              color=[colors[m] for m in methods], edgecolor='white', linewidth=0.8)
for bar, method in zip(bars, methods):
    val = summary[summary['method']==method]['mean_seconds'].values[0]
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02, f'{val:.2f}s',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('(d) Average Generation Time per Sample', fontweight='bold')
ax.set_ylabel('Seconds')
ax.tick_params(axis='x', rotation=15)
ax.grid(alpha=0.3, axis='y')

# (e) Z-score CDF
ax = axes[1, 1]
for method in methods:
    sub = df[df['method'] == method]
    z_sorted = np.sort(sub['z_score'])
    cdf = np.arange(1, len(z_sorted)+1) / len(z_sorted)
    ax.plot(z_sorted, cdf, label=labels[method], color=colors[method], linewidth=2)
ax.axvline(2.0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_title('(e) Z-score Cumulative Distribution', fontweight='bold')
ax.set_xlabel('Z-score')
ax.set_ylabel('Cumulative Probability')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (f) Detection rate vs threshold
ax = axes[1, 2]
thresholds = np.arange(0, 6.1, 0.5)
for method in ['kgw', 'unbiased']:
    sub = df[df['method'] == method]
    rates = [(sub['z_score'] > th).mean() for th in thresholds]
    ax.plot(thresholds, rates, 'o-', label=labels[method], color=colors[method], linewidth=2, markersize=6)
# Baseline as reference
sub_b = df[df['method'] == 'baseline']
baseline_rates = [(sub_b['z_score'] > th).mean() for th in thresholds]
ax.plot(thresholds, baseline_rates, 's--', label=labels['baseline'], color=colors['baseline'], linewidth=1.5, markersize=5, alpha=0.6)
ax.set_title('(f) Detection Rate vs Z Threshold', fontweight='bold')
ax.set_xlabel('Z-score Threshold')
ax.set_ylabel('Detection Rate (Z > threshold)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Assignment 1: KGW vs Unbiased Watermark — Comprehensive Evaluation', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'assignment1_full_charts.png', dpi=180, bbox_inches='tight')
plt.show()
print("Charts saved to assignment1_full_charts.png")


Charts saved to assignment1_full_charts.png


### 4.3 逐 Prompt 分析

分析不同 prompt 类别下 Z-score 的变化，评估水印方法在不同类型文本上的稳定性。

In [6]:
# Category analysis
# Assign category based on prompt content keywords
def categorize_prompt(prompt):
    cats = []
    if any(kw in prompt for kw in ['科学', '物理', '量子', '化学', '基因', '生物', '纳米', '技术']):
        cats.append('Science')
    if any(kw in prompt for kw in ['哲学', '自由意志', '伦理', '社会', '全球化', '文化']):
        cats.append('Humanities')
    if any(kw in prompt for kw in ['编程', '代码', 'Python', 'JavaScript', '算法', '软件']):
        cats.append('Tech')
    if any(kw in prompt for kw in ['诗', '小说', '创作', '文学', '艺术', '散文']):
        cats.append('Creative')
    if any(kw in prompt for kw in ['新闻', '报道', '财经', '经济', '商业', '营销']):
        cats.append('News/Business')
    if any(kw in prompt for kw in ['历史', '文明', '古代', '战争']):
        cats.append('History')
    return cats[0] if cats else 'Other'

df['category'] = df['prompt_id'].apply(lambda pid: categorize_prompt(prompts[pid]))

# Z-score by category and method
cat_analysis = df.groupby(['category', 'method']).agg(
    count=('z_score', 'count'),
    mean_z=('z_score', 'mean'),
    std_z=('z_score', 'std'),
).reset_index()

print("Z-score by Prompt Category and Method:")
display(cat_analysis.round(3).pivot(index='category', columns='method', values='mean_z'))

# Visualize category breakdown
fig, ax = plt.subplots(figsize=(12, 5))
categories = sorted(df['category'].unique())
x = np.arange(len(categories))
width = 0.25

for i, method in enumerate(methods):
    means = [df[(df['category']==cat) & (df['method']==method)]['z_score'].mean() for cat in categories]
    stds = [df[(df['category']==cat) & (df['method']==method)]['z_score'].std() for cat in categories]
    bars = ax.bar(x + i*width, means, width, label=labels[method], color=colors[method], 
                  yerr=stds, capsize=3, edgecolor='white')

ax.axhline(2.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Z=2')
ax.set_xticks(x + width)
ax.set_xticklabels(categories, rotation=20)
ax.set_ylabel('Mean Z-score')
ax.set_title('Mean Z-score by Prompt Category and Method', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT_DIR / 'assignment1_category_analysis.png', dpi=150)
plt.show()


Z-score by Prompt Category and Method:


method,baseline,kgw,unbiased
category,,,
Creative,-0.770,4.944,2.385
History,0.224,5.702,2.795
Humanities,0.350,5.523,2.691
News/Business,0.000,5.143,2.783
Other,0.067,4.919,3.052
Science,0.278,5.235,2.706
Tech,0.000,3.578,2.236


## 5. 结果讨论与结论

### 5.1 主要发现

1. **检测有效性**：KGW 水印产生最强的检测信号，平均 Z-score 显著高于 Unbiased 和 Baseline（p < 0.001）。
   KGW 几乎所有样本的 Z-score 都超过检测阈值 Z=2，而 Unbiased 的检测信号较弱但依然可检测。

2. **文本质量**：水印方法对文本质量（以 PPL 衡量）的影响较小。KGW 的 PPL 略高于 Baseline，
   但差异在统计上不总是显著的。Unbiased 的重加权策略成功将质量损失控制在更小范围内。

3. **计算开销**：三种方法的生成时间差异很小，水印处理器的额外计算开销可忽略不计，
   主要瓶颈在于模型前向传播。

4. **跨类别稳定性**：两种水印方法在不同 prompt 类别上均保持稳定的检测能力，
   说明基于哈希的绿名单机制对文本内容不敏感。

### 5.2 方法对比

| 维度 | KGW | Unbiased |
|------|-----|----------|
| 检测信号强度 | ★★★★★ 很强 | ★★★☆☆ 中等 |
| 文本质量保持 | ★★★★☆ 良好 | ★★★★★ 优秀 |
| 实现复杂度 | ★★★★★ 简单 | ★★★★☆ 较简单 |
| 理论优雅性 | ★★★☆☆ | ★★★★★ |

### 5.3 局限性与未来工作

- 仅在 Qwen2.5-0.5B 模型上验证，需扩展到更多模型尺度
- 缺乏对强释义攻击的鲁棒性评估（见作业 2）
- PPL 作为质量代理指标有局限，需结合人工评估
